# Structured output
Open this notebook from its `Code` folder and select the course Python kernel.
Run the setup, then follow the sections indicated in the slides.

## Setup
Load the course key and create the LangChain client.

In [ ]:
from pathlib import Path
import yaml

key_file = Path("../../../llm_provider_api_key.yaml")
api_key = yaml.safe_load(key_file.read_text())["ollama"]["api_key"]


from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + api_key}},
)

## 1. Sentiment analysis and email classification
A schema describes the response fields. `Literal` lists allowed values.
`with_structured_output` asks the model for an object following that schema.

First, classify the sentiment of a review.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage

class Sentiment(BaseModel):
    sentiment: Literal["POSITIVE", "NEGATIVE", "NEUTRAL"]

sentiment_model = llm.with_structured_output(Sentiment, method="function_calling")
messages = [
    SystemMessage("Return Sentiment. Use POSITIVE, NEGATIVE or NEUTRAL."),
    HumanMessage("I loved the exhibition. I would happily visit again!"),
]
for message in messages:
    message.pretty_print()

In [ ]:
analysis = sentiment_model.invoke(messages)
analysis

In [ ]:
analysis.sentiment

### Format instructions
A model can return the wrong fields or invent a category. We add the schema to the system message to make the expected format explicit.
`model_json_schema` describes the fields and allowed values. We use this pattern in every example below.

In [ ]:
import json
from pprint import pprint

pprint(Sentiment.model_json_schema(), sort_dicts=False)
format_instructions = "Return Sentiment using this schema:\n" + json.dumps(Sentiment.model_json_schema())
SystemMessage(format_instructions).pretty_print()

### Classify an email
The same pattern can assign an email to support, sales or spam.
Inspect the messages before running the call.

In [ ]:
class EmailCategory(BaseModel):
    category: Literal["SUPPORT", "SALES", "SPAM"]

email_model = llm.with_structured_output(EmailCategory, method="function_calling")
format_instructions = "Return EmailCategory using this schema:\n" + json.dumps(EmailCategory.model_json_schema())
messages = [
    SystemMessage("Use SUPPORT for problems, SALES for purchase enquiries, or SPAM for unsolicited advertising.\n" + format_instructions),
    HumanMessage("My order never arrived. Could you help me?"),
]
for message in messages:
    message.pretty_print()

In [ ]:
email_model.invoke(messages)

## 2. Exercise: tune a classifier
The dataset contains 15 requests with correct labels. The museum uses these definitions:
- `TICKETS`: anything about tickets or the ticket office, including prices, discounts, where to buy or collect tickets, and ticket office opening times.
- `DIRECTIONS`: where any other place is or how to reach it, including lifts and stairs.
- `HOURS`: any other question about time, such as opening and closing, tour start times or visit length.

The starting prompt gives the model only the label names. Look at wrong predictions, tune `classification_prompt` and the field `description`, then rerun to get the highest score. The model receives only request text; labels are used afterwards for scoring.

**Tip:** the model cannot guess what each label means for this museum. Try to describe each label better in the prompt.

In [ ]:
data = json.loads(Path("visitor_intents.json").read_text())
pprint(data)

### Tune here
Keep labels and code structure fixed. Rerun this cell after editing the prompt or description.

In [ ]:
class Classification(BaseModel):
    labels: list[Literal["DIRECTIONS", "HOURS", "TICKETS"]] = Field(
        description="One label per request, in the same order."
    )

classifier = llm.with_structured_output(Classification, method="function_calling")
format_instructions = "Return Classification using this schema:\n" + json.dumps(Classification.model_json_schema())
classification_prompt = "Classify the museum requests."

### Classify the requests
Send all 15 requests in one model call. Inspect the messages, then run the scoring cell.

In [ ]:
inputs = [row["text"] for row in data]
count_instruction = f"Return exactly {len(data)} labels: one per request, in order. Use only DIRECTIONS, HOURS or TICKETS. Do not combine requests. Always call Classification."
messages = [
    SystemMessage(classification_prompt + "\n" + format_instructions + "\n" + count_instruction),
    HumanMessage("Requests:\n" + json.dumps(inputs)),
]
for message in messages:
    message.pretty_print()
result = classifier.invoke(messages)

### Inspect errors and improve the score
Accuracy = correct predictions / number of requests. Tune the prompt or description and rerun.

In [ ]:
if len(result.labels) != len(data):
    print("Expected", len(data), "labels, received", len(result.labels), ". Tune the prompt and rerun.")
else:
    correct = 0
    # zip pairs each request with its prediction.
    for row, prediction in zip(data, result.labels):
        if prediction == row["label"]:
            correct += 1
        else:
            print(row["text"], "| expected:", row["label"], "| predicted:", prediction)
    print("Accuracy:", correct / len(data))

## 3. A response that controls the avatar
The server expects `speech_text`, `emotion`, `expression` and `gesture`.
This cell lists the allowed values used by the server.

In [ ]:
Emotion = Literal['NEUTRAL', 'HAPPY', 'ANGRY', 'SAD', 'FEAR', 'DISGUST', 'LOVE', 'SLEEP']
Expression = Literal['😐', '😏', '🙂', '😊', '😀', '😃', '😄', '😁', '😆', '😝', '😂', '😉', '😭', '🥺', '😞', '😔', '😳', '☹️', '😚', '😘', '🥰', '😍', '😡', '😠', '😒', '😱', '😬', '🙄', '👀', '😴']
Gesture = Literal['handup', 'index', 'ok', 'thumbup', 'thumbdown', 'side', 'shrug', 'namaste', 'yes', 'no']

In [ ]:
class ModelReply(BaseModel):
    speech_text: str | None = Field(default=None, description="Text to say.")
    emotion: Emotion | None = Field(default=None, description="Emotion to keep active.")
    expression: Expression | None = Field(default=None, description="Facial expression.")
    gesture: Gesture | None = Field(default=None, description="One gesture. Null keeps the current gesture.")

structured_client = llm.with_structured_output(ModelReply, method="function_calling")

In [ ]:
format_instructions = "Return ModelReply using this schema:\n" + json.dumps(ModelReply.model_json_schema())

### Prepare and inspect the messages
Check the complete prompt before sending it to the model.
Use `null` for an unused expression, emotion or gesture.

In [ ]:
system_prompt = "You are a virtual human. Put spoken words in speech_text. Use only catalog values, or null for unused fields."
request = "Welcome our visitors happily and raise one hand."
messages = [
    SystemMessage(system_prompt + "\n" + format_instructions),
    HumanMessage(request),
]
for message in messages:
    message.pretty_print()

In [ ]:
reply = structured_client.invoke(messages)
reply

### Try on the agent
Open [`langchain_structured_agent.py`](../../../../virtual_human_project/vh_cognitive_architecture_server/agents/Module_1/Chapter_03/langchain_structured_agent.py).
Select its import in `vh_cognitive_server.py`, save and restart the cognitive server:

```python
from agents.Module_1.Chapter_03.langchain_structured_agent import response
```

Ask for a happy welcome and a raised hand. Check the spoken reply and the avatar's behaviour.

## 4. Exercise: short-term conversation memory
Each call receives only the messages we send. Reusing the client does not retain earlier turns.
Start with two separate calls.

In [ ]:
first_message = HumanMessage("My name is Lea. My favourite colour is green.")
first_reply = llm.invoke([first_message])
first_reply.pretty_print()

This question needs information from the earlier turn, but this call sends only the question.

In [ ]:
question = HumanMessage("What is my name and my favourite colour?")
llm.invoke([question]).pretty_print()

### Your task
Use `history.append(...)` to keep the conversation in order.

1. Add `first_message` and `first_reply` to the list.
2. Add `question`, inspect the list, then pass the whole list to `llm.invoke`.
3. Add the new model reply to the list and continue with another question.

Store both human and AI messages. Check that the answer now uses the earlier information.

In [ ]:
history = []
# TODO: accumulate the messages, call the model with history, then save its reply.

### Try on the agent
Complete [`langchain_basic_memory_agent.py`](../../../../virtual_human_project/vh_cognitive_architecture_server/agents/Module_1/Chapter_03/langchain_basic_memory_agent.py), using [`langchain_naive_agent.py`](../../../../virtual_human_project/vh_cognitive_architecture_server/agents/Module_1/Chapter_03/langchain_naive_agent.py) as your starting point.
The file contains only instructions in comments. Write the agent yourself. Keep previous human and AI messages between requests and preserve the system instruction and `AgentReply` return.

After completing the file, select your new agent in `vh_cognitive_server.py`:

```python
from agents.Module_1.Chapter_03.langchain_basic_memory_agent import response
```

Keep only this response import active. Save and restart the cognitive server. Tell the avatar your name and favourite colour, then ask it to recall both. Restarting the server clears this in-memory history.

The list lives in the notebook kernel and is lost on restart.
In Module 2 we will organize conversation memory in graph state and use checkpoints.